## 1. Setup

In [2]:
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/tokenizerv3.pkl .
# !mkdir ./base-models
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/mla-sin.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope-swiglu-moe.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/swiglu-moe-rope-conv.pt ./base-models/

In [3]:
import os, gc, copy, time, math, random, re, json, pickle, zipfile, hashlib, multiprocessing
from dataclasses import dataclass, replace, asdict
from collections import Counter, defaultdict

import numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Data location / field names (check against the EDA cell output) ──
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE7_DIR      = "Phase_7_PCR"
ID_KEY, TEXT_KEY, RELEVANT_KEY = "id", "text", "relevant_candidates"
TOP_K = 7                         # number of candidates evaluated / submitted per query

# True -> tiny subsets + separate output dir, to verify the whole pipeline in minutes
SMOKE_TEST = False
OUT = "./cs781_phase7_smoke" if SMOKE_TEST else "./cs781_phase7"
os.makedirs(OUT, exist_ok=True)

# ── Phase 1 model hyper-parameters (LMConfig reads these) ──
VOCAB_SIZE       = 20_000
TRAIN_BLOCK_SIZE = 2048
N_EMBD           = 768
N_LAYER          = 12
N_HEAD           = 12
N_KV_HEAD        = 4
KV_LATENT_DIM    = N_EMBD // 4

PHASE1_CONFIGS = {
    "gqa-rope":             {"n_layers": 8,  "d_ff": 4 * 768, "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": False, "use_conv": False},
    "mla-sin":              {"n_layers": 12, "d_ff": 4 * 768, "attention_type": "mla", "pos_emb_type": "sinusoidal", "use_moe": False, "use_conv": False},
    "gqa-rope-swiglu-moe":  {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": False},
    "swiglu-moe-rope-conv": {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": True, "local_kernel_size": 25},
}
MAIN_MODEL     = "gqa-rope-swiglu-moe"   # model used for the technique experiments
ABLATION_MODEL = MAIN_MODEL              # set to "gqa-rope" to run the ablations ~2x cheaper (Stage 1 provides its reference run)

# ── Document geometry: chunk = <s> + up to CONTENT_LEN tokens + </s> ──
CHUNK_LEN          = 512
CONTENT_LEN        = CHUNK_LEN - 2
MIN_CHUNK_TOKENS   = 32            # a trailing piece shorter than this is dropped (unless it is the whole document)
DEFAULT_MAX_CHUNKS = 4             # chunks per document fed to the model (the main speed knob)
NUM_TOKENIZE_WORKERS = 4

# ── Training ──
BATCH_QUERIES = 4                  # queries per micro-batch (each brings 1 positive + n_hard negatives; all are in-batch negatives)
GRAD_ACCUM    = 4
EVAL_BATCH_DOCS = 8
QUERY_BLOCK   = 16                 # queries scored against the whole pool at once
GRAD_CKPT     = True               # recompute activations in backward -> much lower memory
WARMUP_FRAC   = 0.06
WEIGHT_DECAY  = 0.01

# ── Hard negatives mined with TF-IDF: take ranks [SKIP, SKIP+POOL) among the non-relevant top-DEPTH candidates ──
TFIDF_DEPTH, HARD_NEG_RANK_SKIP, HARD_NEG_POOL_SIZE = 100, 5, 50
SAVE_TOP = 50                      # ranked candidates saved per query (analysis, R@20 / R@50)

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Tokenizer

In [4]:
class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.reserved = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved); self.stoi = {}; self.merges = []; self.ranks = {}; self.merged_map = {}; self._cache = {}
        # Regex pattern matching words, numbers, individual punctuation, and whitespace sequences
        self.pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        self.pat = re.compile(self.pattern, re.UNICODE)

    def fit(self, texts, max_unique_words=200000):
        """Train a byte-level BPE tokenizer using inverted indexing for fast merging."""
        raw_counts = Counter()
        iterator = [texts] if isinstance(texts, str) else texts

        # Preserve whitespace tokens when populating initial raw_counts
        for text in tqdm(iterator, desc="Building word vocabulary"):
            for chunk in re.findall(self.pattern, str(text), flags=re.UNICODE):
                raw_counts[chunk] += 1

        if max_unique_words and len(raw_counts) > max_unique_words:
            most_common = raw_counts.most_common(max_unique_words)
            raw_counts = dict(most_common)

        word_freqs = {}; pair_counts = defaultdict(int); pair_to_words = defaultdict(set)
        for word, count in raw_counts.items():
            encoded = word.encode("utf-8")
            byte_word = tuple(encoded[i:i+1] for i in range(len(encoded)))
            word_freqs[byte_word] = count

            for i in range(len(byte_word) - 1):
                pair = (byte_word[i], byte_word[i + 1])
                pair_counts[pair] += count
                pair_to_words[pair].add(byte_word)

        base_bytes = sorted({token for word in word_freqs for token in word})
        self.itos = list(self.reserved) + base_bytes; self.merges = []
        target_merges = self.vocab_size - len(self.itos)

        with tqdm(total=target_merges, desc="Training BPE Merges") as pbar:
            while len(self.itos) < self.vocab_size:
                if not pair_counts:
                    break
                best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p[0], p[1]))
                if pair_counts[best_pair] <= 0:
                    break
                merged_token = best_pair[0] + best_pair[1]
                self.merges.append((best_pair, merged_token))
                self.itos.append(merged_token)
                
                words_to_update = list(pair_to_words[best_pair])
                del pair_counts[best_pair]
                del pair_to_words[best_pair]

                p0, p1 = best_pair
                for word in words_to_update:
                    count = word_freqs.get(word, 0)
                    if count == 0:
                        continue

                    new_word = []; i = 0; n = len(word)
                    while i < n:
                        if i + 1 < n and word[i] == p0 and word[i + 1] == p1:
                            new_word.append(merged_token)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1
                    new_word = tuple(new_word)
                    
                    for j in range(len(word) - 1):
                        old_p = (word[j], word[j + 1])
                        if old_p in pair_counts:
                            pair_counts[old_p] -= count
                            if pair_counts[old_p] <= 0:
                                del pair_counts[old_p]

                    for j in range(len(new_word) - 1):
                        new_p = (new_word[j], new_word[j + 1])
                        pair_counts[new_p] += count
                        pair_to_words[new_p].add(new_word)

                    del word_freqs[word]
                    word_freqs[new_word] = count

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self.merged_map = {pair: merged for pair, merged in self.merges}
        self._cache = {}
        return self

    def encode_word(self, word):
        """Encode single word using memoization and local pair updates."""
        if word in self._cache:
            return self._cache[word]

        encoded = word.encode("utf-8")
        tokens = [encoded[i:i+1] for i in range(len(encoded))]

        if len(tokens) <= 1:
            res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
            self._cache[word] = res
            return res

        while len(tokens) > 1:
            min_rank = float("inf")
            min_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.ranks.get(pair, float("inf"))
                if rank < min_rank:
                    min_rank = rank
                    min_pair = pair

            if min_pair is None or min_rank == float("inf"):
                break

            merged = self.merged_map[min_pair]
            new_tokens = []
            i = 0
            n = len(tokens)
            p0, p1 = min_pair

            while i < n:
                if i + 1 < n and tokens[i] == p0 and tokens[i + 1] == p1:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        self._cache[word] = res
        return res

    def encode(self, text):
        """Encode a sentence while preserving original spaces and newlines."""
        ids = [self.stoi["<s>"]]
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        for chunk in re.findall(pattern, str(text), flags=re.UNICODE):
            ids.extend(self.encode_word(chunk))
        ids.append(self.stoi["</s>"])
        return ids
    
    def decode(self, ids):
        """Decode BPE token IDs back into text seamlessly."""
        byte_list = []
        for idx in ids:
            tok = self.itos[idx] if 0 <= idx < len(self.itos) else "<unk>"
            if tok not in self.reserved and isinstance(tok, bytes):
                byte_list.append(tok)
        try:
            return b"".join(byte_list).decode("utf-8", errors="ignore")
        except Exception:
            return ""

    def get_vocab_size(self):
        return self.vocab_size

def train_tokenizer(train_text: str, vocab_size: int):
    """Train A-1 BPE tokenizer on `train_text` and return the trained tokenizer."""
    tokenizer = ByteLevelBPE(vocab_size)
    tokenizer.fit(train_text)
    return tokenizer
    
# Load Pretrained Tokenizer
with open("tokenizerv3.pkl", "rb") as f:
    tokenizer = pickle.load(f)
BOS, PAD, UNK, EOS = (tokenizer.stoi[t] for t in ("<s>", "<pad>", "<unk>", "</s>"))
assert len(tokenizer.itos) == VOCAB_SIZE
print("tokenizer vocab:", len(tokenizer.itos))

tokenizer vocab: 20000


## 3. Model Class Definitions

In [5]:
@dataclass
class LMConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = N_EMBD
    n_layers: int = N_LAYER
    n_heads: int = N_HEAD
    n_kv_heads: int = N_KV_HEAD
    kv_latent_dim: int = KV_LATENT_DIM
    d_ff: int = 1408          # For SwiGLU (2/3) * 4 * d_model => 2/3 * 2048 = 1365 --> round to nearest upper multiple of 64 (1408)
    max_seq_len: int  = TRAIN_BLOCK_SIZE
    dropout: float = 0.0            # ~1 pass over the data -> no need for dropout
    pos_emb_type: str = "rope"      # "rope" or "sinusoidal"
    attention_scale: float = None
    norm_first: bool = True         # FIX: pre-LN (post-LN + lr 1e-3 + no warm-up is unstable)
    n_experts:  int   = 4
    top_k:      int   = 2
    aux_loss_coef: float = 0.01
    attention_type: str = "gqa"
    use_moe: bool = False, 
    use_conv: bool = False
    local_kernel_size: int = 25
    
# Rotary Positional Embeddings
def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """ Returns (cos, sin) : the split-half rotate_half convention: channel i is paired with channel i+d_head/2. """
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    ang = torch.cat([torch.outer(torch.arange(max_seq_len).float(), inv)] * 2, -1)
    return ang.cos(), ang.sin()
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree rotation step used by the "rotate_half" formulation of RoPE."""
    x1, x2 = x[ ..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 : ]
    return torch.cat((-x2, x1), dim = -1)
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """ x: (B, n_heads, T, d_head) -- a query or key tensor. cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T. Returns the rotated tensor, same shape as x. Formula: x_rotated = x * cos + rotate_half(x) * sin """
    cos, sin = (cos[None, None], sin[None, None]) if cos.dim() == 2 else (cos[:, None], sin[:, None])
    return x.float() * cos + rotate_half(x.float()) * sin

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)
    angle = pos / (10000 ** (i / d_model))
    return torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1).reshape(max_len, d_model)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=True, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep, self.d_head = n_heads // n_kv_heads, d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[position_ids], self.rope_sin[position_ids])
            k = apply_rope(k, self.rope_cos[position_ids], self.rope_sin[position_ids])
             
        # Append to KV cache if provided
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        new_kv_cache = (k, v) if use_cache else None

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # Only set is_causal=True if processing multiple prompt tokens at once (T > 1)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0,  is_causal=is_causal, scale=self.attention_scale)
        out = self.resid_dropout(self.out_proj(context.transpose(1, 2).reshape(B, T, C)))
        
        return (out, new_kv_cache) if use_cache else out

class MultiHeadLatentAttention(nn.Module):
    """Joint KV Compression Multi-Head Latent Attention module drop-in replacement for CausalSelfAttention."""

    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, dropout=0.0, attention_scale=None):
        super().__init__()
        assert (d_model % n_heads == 0 and 0 < kv_latent_dim < d_model), "d_model must be divisible by n_heads, and 0 < kv_latent_dim < d_model"

        self.n_heads, self.kv_latent_dim, self.d_head = n_heads, kv_latent_dim, d_model // n_heads
        self.attention_scale = ((1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale))

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_down_proj = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.k_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        # Dropouts
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        """Forward pass supporting sequence generation with KV cache and FlashAttention/SDPA."""
        B, T, C = x.shape
        q = (self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T, d_head)
        c_kv = self.kv_down_proj(x)  # (B, T, kv_latent_dim)

        if kv_cache is not None:
            past_c_kv = (kv_cache[0] if isinstance(kv_cache, (tuple, list)) else kv_cache)
            c_kv = torch.cat([past_c_kv, c_kv], dim=1)  # Append along sequence dim T
        new_kv_cache = (c_kv,) if use_cache else None

        k = (self.k_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        v = (self.v_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=is_causal, scale=self.attention_scale)
        out = context.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        return (out, new_kv_cache) if use_cache else out

### Model Definitions
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)
        
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up   = nn.Linear(d_model, d_ff, bias=False); self.down = nn.Linear(d_ff, d_model, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class TopKRouter(nn.Module):
    def __init__(self, d_model: int, n_experts: int, top_k: int):
        super().__init__()
        self.top_k  = top_k; self.linear = nn.Linear(d_model, n_experts, bias=False)
    def forward(self, x_flat):                           
        logits = self.linear(x_flat); topk_vals, topk_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)            # (B*T, top_k)
        dispatch = torch.zeros_like(logits, dtype=weights.dtype)
        dispatch.scatter_(-1, topk_idx, weights)          # (B*T, n_experts)
        return dispatch, logits                           # logits kept for aux loss

class MoEFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k, dropout, aux_coef=0.01):
        super().__init__()
        self.router   = TopKRouter(d_model, n_experts, top_k)
        self.experts  = nn.ModuleList([SwiGLUFeedForward(d_model, d_ff, dropout) for _ in range(n_experts)])
        self.aux_coef = aux_coef; self.aux_loss = torch.tensor(0.0)

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)
        dispatch, router_logits = self.router(x_flat)

        z = torch.logsumexp(router_logits, dim=-1)
        self.aux_loss = self.aux_coef * (z ** 2).mean()
        out = torch.zeros_like(x_flat)
        for e_idx, expert in enumerate(self.experts):
            w  = dispatch[:, e_idx];  mask = w > 0
            if not mask.any():
                continue
            out[mask] += w[mask].unsqueeze(-1) * expert(x_flat[mask])

        return out.reshape(B, T, D)

class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = cfg.norm_first; self.ln1 = nn.LayerNorm(cfg.d_model); self.ln2 = nn.LayerNorm(cfg.d_model)
        if(cfg.attention_type == "mla"):
            self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        else:
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"), attention_scale=cfg.attention_scale)
        if(cfg.use_moe):
            self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        else:
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        
    def _attn(self, h, position_ids, attn_mask, kv_cache, use_cache):
        r = self.attn(h, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        return r if use_cache else (r, None)
        
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        if self.norm_first:
            a, new_cache = self._attn(self.ln1(x), position_ids, attn_mask, kv_cache, use_cache)
            x = x + a
            x = x + self.ffn(self.ln2(x))
        else:
            a, new_cache = self._attn(x, position_ids, attn_mask, kv_cache, use_cache)
            x = self.ln1(x + a)
            x = self.ln2(x + self.ffn(x))
        return x, new_cache
        
class LocalDepthwiseConv(nn.Module):
    """ Depthwise causal convolution over the sequence dimension. Acts on each channel independently — O(T * C * kernel) cost. kernel_size = 25 covers ~25 tokens of local context. """
    def __init__(self, d_model: int, kernel_size: int = 25, dropout: float = 0.0):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel_size, padding=0, groups=d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.conv.weight, 0.0, 0.02)
        nn.init.normal_(self.proj.weight, 0.0, 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, C) → (B, T, C)"""
        B, T, C = x.shape
        xc = x.transpose(1, 2)
        xc = F.pad(xc, (self.kernel_size - 1, 0))
        xc = self.conv(xc)          # (B, C, T)  — output length = T (causal)
        xc = xc.transpose(1, 2)    # (B, T, C)
        return self.drop(self.proj(F.gelu(xc)))

class ParallelLocalGlobalBlock(nn.Module):
    """Pre-LN block with global attention and local conv running IN PARALLEL."""
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.RMSNorm(cfg.d_model); self.ln2  = nn.RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=cfg.pos_emb_type == "rope", attention_scale=cfg.attention_scale)
        self.conv = LocalDepthwiseConv(cfg.d_model, kernel_size=cfg.local_kernel_size, dropout=cfg.dropout)
        self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        self.alpha = nn.Parameter(torch.ones(1))    # global attention scale
        self.beta  = nn.Parameter(torch.full((1,), 0.1))  # local conv scale

    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        normed = self.ln1(x)
        attn_out = self.attn(normed, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        attn_out, new_cache = attn_out if use_cache else attn_out, None
        conv_out = self.conv(normed)
        
        x = x + self.alpha * attn_out + self.beta * conv_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache

    @property
    def aux_loss(self):
        return self.ffn.aux_loss


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.use_sin = cfg.pos_emb_type == "sinusoidal"
        self.emb_scale = math.sqrt(cfg.d_model) if self.use_sin else 1.0
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if self.use_sin:
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        if cfg.use_conv:
            self.blocks = nn.ModuleList([ParallelLocalGlobalBlock(cfg) for _ in range(cfg.n_layers)])
        else:
            self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self._init_weights()
        self.head.weight = self.tok_emb.weight # tie AFTER init so the embedding init survives

    def aux_loss(self):
        return sum(blk.ffn.aux_loss for blk in self.blocks)
        
    def _init_weights(self):
        emb_std = self.cfg.d_model ** -0.5 if self.use_sin else 0.02
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=emb_std)
        for n, p in self.named_parameters():
            if n.endswith("attn.out_proj.weight") or n.endswith("ffn.net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.cfg.n_layers))
    
    def forward(self, idx: torch.Tensor, kv_cache_list=None, use_cache=False, attention_mask=None) -> torch.Tensor:
        """attention_mask: optional (B, past+T) tensor, 1 = real token, 0 = left padding."""
        B, T = idx.shape
        past = kv_cache_list[0][0].shape[-2] if kv_cache_list is not None else 0
        S = past + T

        if attention_mask is None:
            position_ids = torch.arange(past, S, device=idx.device).unsqueeze(0).expand(B, T)
            attn_mask = None                                   # fast path: is_causal=True inside SDPA
        else:
            position_ids = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, past:S]
            q_idx = torch.arange(past, S, device=idx.device)[:, None]
            k_idx = torch.arange(S, device=idx.device)[None, :]
            allowed = (k_idx <= q_idx)[None] & attention_mask[:, None, :S].bool()   # causal & not-padding
            allowed = allowed | (k_idx == q_idx)[None]         # pad rows attend to themselves (no NaNs)
            attn_mask = allowed.unsqueeze(1)                   # (B, 1, T, S)

        x = self.tok_emb(idx) * self.emb_scale
        if self.use_sin:
            x = x + self.pos_emb[position_ids]

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache_list[i] if kv_cache_list is not None else None
            x, c = block(x, position_ids, attn_mask, layer_cache, use_cache)
            new_caches.append(c)

        logits = self.head(self.ln_f(x))
        return (logits, new_caches) if use_cache else logits
        
    @torch.no_grad()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=384, temperature=0.7, top_p=0.9,
                 repetition_penalty=1.0, banned_token_ids=(0, 1, 2, 3), do_sample=True):
        self.eval()
        out = input_ids
        if attention_mask is None:
            attention_mask = torch.ones_like(out)
    
        logits, kv = self(out, use_cache=True, attention_mask=attention_mask)      # prefill
        next_logits = logits[:, -1, :].float()
    
        for step in range(max_new_tokens):
            l = next_logits.clone()
            if repetition_penalty != 1.0:
                sc = l.gather(1, out); sc = torch.where(sc < 0, sc * repetition_penalty, sc / repetition_penalty)
                l = l.scatter(1, out, sc)
            if banned_token_ids:                       # never emit <pad>/<unk>/<s>/</s>: we need 128 words
                l[:, list(banned_token_ids)] = float("-inf")

            if do_sample and temperature > 0.0:
                l = l / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(l, descending=True)
                    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    l = l.masked_fill(remove.scatter(1, sorted_idx, remove), float("-inf"))
                next_token = torch.multinomial(F.softmax(l, dim=-1), num_samples=1)
            else:
                next_token = l.argmax(dim=-1, keepdim=True)

            out = torch.cat([out, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
            if step == max_new_tokens - 1:
                break
            logits, kv = self(next_token, kv_cache_list=kv, use_cache=True, attention_mask=attention_mask)
            next_logits = logits[:, -1, :].float()
        return out

## 4. Data Loading

In [6]:
def load_phase1_model(model_name):
    cfg = LMConfig()
    for attr, val in PHASE1_CONFIGS[model_name].items():
        setattr(cfg, attr, val)

    checkpoint_data = torch.load(f"./base-models/{model_name}.pt", map_location="cpu", weights_only=False)
    base_lm = DecoderLM(cfg)
    base_lm.load_state_dict(checkpoint_data["model"], strict=True)
    del checkpoint_data; gc.collect()
    return base_lm

@dataclass
class DocSet:
    ids: list                    # IndianKanoon case ids (strings), in file order
    texts: list                  # cleaned case text (sentences joined)
    fingerprints: list           # md5 of cleaned text (token cache key, self / duplicate detection)
    token_ids: list              # np.int32 array per case, WITHOUT <s>/</s> (filled in Cell 7)

@dataclass
class PCRSplit:
    name: str
    queries: DocSet
    pool: DocSet                 # candidate pool that belongs to THIS split
    gold_rows: list              # per query: np.int64 array of relevant pool rows (None for test)
    self_rows: list              # per query: pool rows that ARE the query (same id or identical text); never ranked

def clean_text(text):
    """Light cleanup. Keep consistent with whatever was applied to the Phase 1 pretraining text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse many blank lines
    return text.strip()

def fingerprint(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def read_json(relative_path):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=f"{PHASE7_DIR}/{relative_path}")
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    if isinstance(records, dict):                          # <-- ADD THIS
        records = [{ID_KEY: k, **v} for k, v in records.items()]  # <-- ADD THIS
    assert isinstance(records, list), f"{relative_path}: expected a list of records, got {type(records)}"
    return records

def sentences_to_text(sentences, where):
    assert isinstance(sentences, list) and all(isinstance(s, str) for s in sentences), f"{where}: expected a list of sentences"
    return clean_text(" ".join(sentences))

def build_docset(records, where):
    ids = [r[ID_KEY] for r in records]
    assert all(isinstance(i, str) for i in ids), f"{where}: ids must be strings (leading zeros matter)"
    assert len(set(ids)) == len(ids), f"{where}: duplicate ids"
    texts = [sentences_to_text(r[TEXT_KEY], f"{where} {r[ID_KEY]}") for r in records]
    assert all(len(t) > 0 for t in texts), f"{where}: empty text after cleaning"
    return DocSet(ids, texts, [fingerprint(t) for t in texts], None)

def build_split(name, folder, has_labels):
    query_records = read_json(f"{folder}/{name}.json")
    pool = build_docset(read_json(f"{folder}/{name}_candidates.json"), f"{name} pool")
    queries = build_docset(query_records, f"{name} queries")
    assert len(pool.ids) >= SAVE_TOP, f"{name}: candidate pool has only {len(pool.ids)} documents"
    pool_row = {case_id: row for row, case_id in enumerate(pool.ids)}

    rows_by_text = defaultdict(list)
    for row, fp in enumerate(pool.fingerprints):
        rows_by_text[fp].append(row)
    self_rows = []
    for qid, fp in zip(queries.ids, queries.fingerprints):
        rows = set(rows_by_text.get(fp, []))
        if qid in pool_row:
            rows.add(pool_row[qid])
        self_rows.append(np.array(sorted(rows), dtype=np.int64))

    gold_rows = None
    if has_labels:
        gold_rows = []
        for r in query_records:
            relevant = r[RELEVANT_KEY]
            assert isinstance(relevant, list) and all(isinstance(c, str) for c in relevant), f"{name}: bad relevant list for {r[ID_KEY]}"
            outside = [c for c in relevant if c not in pool_row]
            assert not outside, f"{name}: query {r[ID_KEY]} lists relevant ids that are not in its own candidate pool: {outside[:3]}"
            gold_rows.append(np.array(sorted({pool_row[c] for c in relevant}), dtype=np.int64))
        for q, (gold, own) in enumerate(zip(gold_rows, self_rows)):
            assert len(np.intersect1d(gold, own)) == 0, f"{name}: query {queries.ids[q]} is listed as relevant to itself"
    return PCRSplit(name, queries, pool, gold_rows, self_rows)

def select_queries(split, rows):
    rows = list(rows)
    q = split.queries
    pick = lambda values: None if values is None else [values[i] for i in rows]
    return PCRSplit(split.name, DocSet(pick(q.ids), pick(q.texts), pick(q.fingerprints), pick(q.token_ids)),
                    split.pool, pick(split.gold_rows), pick(split.self_rows))

def restrict_pool(split, keep_rows):
    """Smoke test only: shrink the candidate pool (keeps all relevant rows) and remap row indices."""
    keep = sorted(set(int(r) for r in keep_rows))
    remap = {old: new for new, old in enumerate(keep)}
    p = split.pool
    pool = DocSet([p.ids[i] for i in keep], [p.texts[i] for i in keep], [p.fingerprints[i] for i in keep], None)
    gold = None if split.gold_rows is None else [np.array([remap[int(r)] for r in g], dtype=np.int64) for g in split.gold_rows]
    own = [np.array([remap[int(r)] for r in s if int(r) in remap], dtype=np.int64) for s in split.self_rows]
    return PCRSplit(split.name, split.queries, pool, gold, own)

train_split = build_split("train", "Train", True)
dev_split   = build_split("dev",   "Dev",   True)
test_split  = build_split("test",  "Test",  False)
for s in (train_split, dev_split, test_split):
    print(f"raw {s.name}: {len(s.queries.ids)} queries | pool {len(s.pool.ids)}")

# 1) identical query texts inside train: keep the first, merge (union) the relevant sets
first_row, keep, merged = {}, [], [g.copy() for g in train_split.gold_rows]
for row, fp in enumerate(train_split.queries.fingerprints):
    if fp in first_row:
        merged[first_row[fp]] = np.union1d(merged[first_row[fp]], train_split.gold_rows[row])
    else:
        first_row[fp] = row; keep.append(row)
print(f"train: merged {len(train_split.queries.ids) - len(keep)} duplicate query texts")
train_split.gold_rows = merged
train_split = select_queries(train_split, keep)

# 2) train queries without any relevant candidate cannot form a training pair
with_gold = [q for q, g in enumerate(train_split.gold_rows) if len(g) > 0]
print(f"train: removed {len(train_split.queries.ids) - len(with_gold)} queries with no relevant candidate")
train_split = select_queries(train_split, with_gold)

# 3) dev queries that also are train queries (same id or same text) would inflate model selection
train_ids, train_fps = set(train_split.queries.ids), set(train_split.queries.fingerprints)
leaky = {i for i, (qid, fp) in enumerate(zip(dev_split.queries.ids, dev_split.queries.fingerprints)) if qid in train_ids or fp in train_fps}
print(f"dev: removed {len(leaky)} queries that also occur in train")
dev_split = select_queries(dev_split, [i for i in range(len(dev_split.queries.ids)) if i not in leaky])

# 4) test is never filtered (one ranking per test query is required); only report overlap
print("test queries also in train:", sum(qid in train_ids or fp in train_fps for qid, fp in zip(test_split.queries.ids, test_split.queries.fingerprints)))

if SMOKE_TEST:
    rng = np.random.RandomState(SEED)
    def shrink(split, n_queries):
        split = select_queries(split, rng.choice(len(split.queries.ids), n_queries, replace=False))
        needed = set() if split.gold_rows is None else {int(r) for g in split.gold_rows for r in g}
        return restrict_pool(split, needed | set(rng.choice(len(split.pool.ids), 80, replace=False).tolist()))
    train_split, dev_split, test_split = shrink(train_split, 24), shrink(dev_split, 8), shrink(test_split, 8)
for s in (train_split, dev_split, test_split):
    print(f"final {s.name}: {len(s.queries.ids)} queries | pool {len(s.pool.ids)}")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

raw train: 827 queries | pool 4320
raw dev: 118 queries | pool 1023
raw test: 237 queries | pool 1727
train: merged 0 duplicate query texts
train: removed 10 queries with no relevant candidate
dev: removed 0 queries that also occur in train
test queries also in train: 1
final train: 817 queries | pool 4320
final dev: 118 queries | pool 1023
final test: 237 queries | pool 1727


In [7]:
TOKEN_CACHE_PATH = f"{OUT}/token_cache_tokenizerv3_{VOCAB_SIZE}.pkl"
token_cache = pickle.load(open(TOKEN_CACHE_PATH, "rb")) if os.path.exists(TOKEN_CACHE_PATH) else {}

def _encode_text(text):
    """Token ids of the whole case WITHOUT <s>/</s> (every chunk gets its own markers)."""
    return np.array(tokenizer.encode(text)[1:-1], dtype=np.int32)

all_sets = [s for split in (train_split, dev_split, test_split) for s in (split.queries, split.pool)]
total_chars = sum(len(t) for s in all_sets for t in s.texts)
sample = train_split.pool.texts[:20]
t0 = time.perf_counter(); [_encode_text(t) for t in sample]
rate = sum(len(t) for t in sample) / (time.perf_counter() - t0)
print(f"tokenizer ~{rate / 1e3:.0f}k chars/s on one process -> roughly {total_chars / rate / NUM_TOKENIZE_WORKERS / 60:.0f} min "
      f"for {total_chars / 1e6:.0f}M chars with {NUM_TOKENIZE_WORKERS} workers (one-off, cached on disk)")

def tokenize_docset(docset, name):
    pending = {}
    for fp, text in zip(docset.fingerprints, docset.texts):
        if fp not in token_cache:
            pending[fp] = text
    if pending:
        fps = list(pending)
        with multiprocessing.get_context("fork").Pool(NUM_TOKENIZE_WORKERS) as workers:
            results = workers.imap(_encode_text, [pending[fp] for fp in fps], chunksize=8)
            for fp, tokens in zip(fps, tqdm(results, total=len(fps), desc=f"tokenize {name}")):
                token_cache[fp] = tokens
    docset.token_ids = [token_cache[fp] for fp in docset.fingerprints]
    assert all(len(t) > 0 for t in docset.token_ids), f"{name}: a document has no tokens"

for split in (train_split, dev_split, test_split):
    tokenize_docset(split.queries, f"{split.name} queries")
    tokenize_docset(split.pool, f"{split.name} pool")
pickle.dump(token_cache, open(TOKEN_CACHE_PATH, "wb"))

# sanity: byte-level BPE should almost never emit <unk>
sample_tokens = np.concatenate(train_split.pool.token_ids[:200])
print("unk fraction in train pool tokens:", (sample_tokens == UNK).mean())

def doc_chunk_starts(num_tokens):
    """Non-overlapping windows of CONTENT_LEN tokens; a tiny trailing window is dropped."""
    starts = [s for s in range(0, num_tokens, CONTENT_LEN) if num_tokens - s >= MIN_CHUNK_TOKENS]
    return starts if starts else [0]

REGION_THIRDS = {"first_third": 0, "middle_third": 1, "last_third": 2}   # used for the section probe

def pick_evenly(candidates, k):
    if len(candidates) <= k:
        return candidates
    positions = np.unique(np.linspace(0, len(candidates) - 1, k).round().astype(int))
    return [candidates[p] for p in positions]

def select_chunk_indices(num_chunks, strategy, max_chunks):
    """Which chunks of a document are fed to the model."""
    all_idx = list(range(num_chunks))
    if strategy == "head":
        return all_idx[:max_chunks]
    if strategy == "tail":
        return all_idx[-max_chunks:]
    if strategy == "head_tail":                                  # beginning + end of the case
        if num_chunks <= max_chunks:
            return all_idx
        n_head = max_chunks // 2
        return all_idx[:n_head] + all_idx[-(max_chunks - n_head):]
    if strategy == "uniform":                                    # evenly spaced over the whole case
        return pick_evenly(all_idx, max_chunks)
    if strategy in REGION_THIRDS:                                # only one third of the case
        third = REGION_THIRDS[strategy]
        lo = num_chunks * third // 3
        hi = max(num_chunks * (third + 1) // 3, lo + 1)
        return pick_evenly(all_idx[lo:hi], max_chunks)
    raise ValueError(f"unknown chunk strategy: {strategy}")

def doc_chunks(tokens, strategy, max_chunks):
    starts = doc_chunk_starts(len(tokens))
    chosen = select_chunk_indices(len(starts), strategy, max_chunks)
    return [[BOS] + tokens[starts[c]: starts[c] + CONTENT_LEN].tolist() + [EOS] for c in chosen]

def pad_chunks(chunk_lists, n_chunks=None):
    """list (per document) of chunk lists -> ids [B, C, L] and chunk_mask [B, C]."""
    n_chunks = n_chunks or max(len(chunks) for chunks in chunk_lists)
    max_len = max(len(c) for chunks in chunk_lists for c in chunks)
    ids = torch.full((len(chunk_lists), n_chunks, max_len), PAD, dtype=torch.long)
    chunk_mask = torch.zeros(len(chunk_lists), n_chunks, dtype=torch.bool)
    for b, chunks in enumerate(chunk_lists):
        for c, chunk in enumerate(chunks):
            ids[b, c, :len(chunk)] = torch.tensor(chunk)
            chunk_mask[b, c] = True
    return ids, chunk_mask

class DocDataset(Dataset):
    """Inference: one item = the selected chunks of one document."""
    def __init__(self, docset, strategy, max_chunks):
        self.docset, self.strategy, self.max_chunks = docset, strategy, max_chunks
    def __len__(self):
        return len(self.docset.ids)
    def __getitem__(self, i):
        return doc_chunks(self.docset.token_ids[i], self.strategy, self.max_chunks)

def tfidf_hard_pool(split):
    """Per training query: non-relevant pool rows ranked high by TF-IDF (ranks SKIP..SKIP+POOL). Cached on disk."""
    path = f"{OUT}/hardneg_tfidf_train.pkl"
    if os.path.exists(path):
        saved = pickle.load(open(path, "rb"))
        assert saved["query_ids"] == split.queries.ids and saved["pool_size"] == len(split.pool.ids), "cached hard negatives belong to another train set"
        return saved["pools"]
    ranked = tfidf_rank(split.queries.texts, split.pool.texts, split.self_rows, TFIDF_DEPTH)
    pools = []
    for q, row in enumerate(ranked):
        gold = set(split.gold_rows[q].tolist())
        non_gold = [int(r) for r in row if int(r) not in gold]
        pools.append(np.array(non_gold[HARD_NEG_RANK_SKIP: HARD_NEG_RANK_SKIP + HARD_NEG_POOL_SIZE], dtype=np.int64))
    pickle.dump({"query_ids": split.queries.ids, "pool_size": len(split.pool.ids), "pools": pools}, open(path, "wb"))
    return pools

class PairDataset(Dataset):
    """One item = one training query with a random relevant case and sampled negatives (row indices only)."""
    def __init__(self, split, exp):
        self.split, self.n_hard, self.mode = split, exp.n_hard, exp.hard_neg
        self.hard_pool = tfidf_hard_pool(split) if exp.hard_neg == "tfidf" else None
        if self.hard_pool is not None:
            assert min(len(p) for p in self.hard_pool) >= exp.n_hard, "TF-IDF hard-negative pool smaller than n_hard"
    def __len__(self):
        return len(self.split.queries.ids)
    def __getitem__(self, q):
        positive = int(random.choice(self.split.gold_rows[q]))
        if self.mode == "none":
            negatives = []
        elif self.mode == "tfidf":
            negatives = [int(r) for r in random.sample(list(self.hard_pool[q]), self.n_hard)]
        else:
            assert self.mode == "random", self.mode
            forbidden = set(self.split.gold_rows[q].tolist()) | set(self.split.self_rows[q].tolist())
            negatives = []
            while len(negatives) < self.n_hard:
                row = random.randrange(len(self.split.pool.ids))
                if row not in forbidden and row not in negatives:
                    negatives.append(row)
        return {"q": q, "pos": positive, "neg": negatives}

class PairBatcher:
    """Builds the tensors of a contrastive batch. Documents are de-duplicated across the batch; `not_negative[q, d]`
    marks pool documents that must not be used as negatives for query q (all its relevant cases and its own copy)."""
    def __init__(self, split, exp):
        self.split, self.strategy, self.max_chunks = split, exp.strategy, exp.max_chunks
    def __call__(self, items):
        doc_rows = []
        for it in items:
            for row in [it["pos"]] + it["neg"]:
                if row not in doc_rows:
                    doc_rows.append(row)
        doc_index = {row: i for i, row in enumerate(doc_rows)}
        not_negative = torch.zeros(len(items), len(doc_rows), dtype=torch.bool)
        for i, it in enumerate(items):
            excluded = np.concatenate([self.split.gold_rows[it["q"]], self.split.self_rows[it["q"]]])
            not_negative[i] = torch.from_numpy(np.isin(doc_rows, excluded))
        q_ids, q_mask = pad_chunks([doc_chunks(self.split.queries.token_ids[it["q"]], self.strategy, self.max_chunks) for it in items])
        d_ids, d_mask = pad_chunks([doc_chunks(self.split.pool.token_ids[r], self.strategy, self.max_chunks) for r in doc_rows])
        return {"q_ids": q_ids, "q_mask": q_mask, "d_ids": d_ids, "d_mask": d_mask,
                "target": torch.tensor([doc_index[it["pos"]] for it in items]), "not_negative": not_negative,
                "q_rows": torch.tensor([it["q"] for it in items]), "d_rows": torch.tensor(doc_rows)}

tokenizer ~3141k chars/s on one process -> roughly 0 min for 325M chars with 4 workers (one-off, cached on disk)
unk fraction in train pool tokens: 6.308875696633597e-05


## 5. Metrics

In [8]:
def rank_from_scores(scores, self_rows, top_k):
    """scores [n_queries, pool] -> ranked pool rows [n_queries, top_k]; a query's own copy is never ranked."""
    masked = torch.from_numpy(np.array(scores, dtype=np.float32))
    for q, rows in enumerate(self_rows):
        masked[q, torch.from_numpy(rows)] = float("-inf")
    return torch.topk(masked, min(top_k, masked.size(1)), dim=1).indices.numpy()

def retrieval_metrics(ranked, gold_rows):
    """Micro-F1@7 / P@7 / R@7 exactly as the competition defines them (gold sets are neither truncated nor padded)."""
    total_gold = sum(len(g) for g in gold_rows)
    assert total_gold > 0, "no relevant candidates at all"
    hits = lambda k: sum(len(np.intersect1d(ranked[q, :k], g)) for q, g in enumerate(gold_rows))
    tp = hits(TOP_K)
    precision, recall = tp / (TOP_K * len(gold_rows)), tp / total_gold
    f1 = 2 * precision * recall / (precision + recall) if tp > 0 else 0.0
    return {"F1@7": f1, "P@7": precision, "R@7": recall, "R@20": hits(20) / total_gold, "R@50": hits(SAVE_TOP) / total_gold}

def max_possible_f1(gold_rows):
    """Upper bound of Micro-F1@7 (a query with more than 7 relevant cases cannot be fully retrieved)."""
    tp = sum(min(TOP_K, len(g)) for g in gold_rows)
    precision, recall = tp / (TOP_K * len(gold_rows)), tp / sum(len(g) for g in gold_rows)
    return 2 * precision * recall / (precision + recall)

def tfidf_rank(query_texts, pool_texts, self_rows, top_k):
    """Lexical ranking: TF-IDF cosine. Used ONLY as a baseline and to mine hard negatives (statistics of the provided text, no pretrained model)."""
    vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.8, dtype=np.float32)
    vectorizer.fit(pool_texts + query_texts)
    pool_matrix, query_matrix = vectorizer.transform(pool_texts), vectorizer.transform(query_texts)
    ranked = []
    for start in range(0, len(query_texts), 128):
        sims = (query_matrix[start: start + 128] @ pool_matrix.T).toarray()
        ranked.append(rank_from_scores(sims, self_rows[start: start + 128], top_k))
    return np.concatenate(ranked)

## 6. EDA

In [9]:
pct = [50, 75, 90, 95, 99, 100]
for split in (train_split, dev_split, test_split):
    q_len = np.array([len(t) for t in split.queries.token_ids]); p_len = np.array([len(t) for t in split.pool.token_ids])
    print(f"\n== {split.name}: {len(q_len)} queries | pool {len(p_len)} documents")
    print("  tokens/query percentiles (50/75/90/95/99/max):", np.percentile(q_len, pct).astype(int))
    print("  tokens/pool-doc percentiles:                 ", np.percentile(p_len, pct).astype(int))
    p_chunks = np.array([len(doc_chunk_starts(n)) for n in p_len])
    print(f"  pool docs that fit entirely in {DEFAULT_MAX_CHUNKS} chunks: {(p_chunks <= DEFAULT_MAX_CHUNKS).mean():.1%} | median chunks/doc {np.median(p_chunks):.0f}")
    print("  queries whose own copy is in their pool:", sum(len(s) > 0 for s in split.self_rows))
    if split.gold_rows is not None:
        n_gold = np.array([len(g) for g in split.gold_rows])
        print("  relevant/query percentiles (min/25/50/75/90/max):", np.percentile(n_gold, [0, 25, 50, 75, 90, 100]).astype(int))
        print(f"  share of the pool that is relevant to a query: {n_gold.mean() / len(p_len):.4%} | queries with more than {TOP_K} relevant: {(n_gold > TOP_K).mean():.1%}")
        print(f"  upper bound of Micro-F1@{TOP_K}: {max_possible_f1(split.gold_rows):.4f}")
        popularity = Counter(int(r) for g in split.gold_rows for r in g)
        print("  most frequently relevant pool docs (count):", [c for _, c in popularity.most_common(5)], f"| distinct relevant docs: {len(popularity)}")

pool_fps = {s.name: set(s.pool.fingerprints) for s in (train_split, dev_split, test_split)}
print("\npool text overlap train/dev:", len(pool_fps["train"] & pool_fps["dev"]), "| train/test:", len(pool_fps["train"] & pool_fps["test"]), "| dev/test:", len(pool_fps["dev"] & pool_fps["test"]))
print("dev queries whose text is in the train pool:", sum(fp in pool_fps["train"] for fp in dev_split.queries.fingerprints))
print("test queries whose text is in the train pool:", sum(fp in pool_fps["train"] for fp in test_split.queries.fingerprints))

# Lexical reference: anything trained should beat this
ranked_tfidf = tfidf_rank(dev_split.queries.texts, dev_split.pool.texts, dev_split.self_rows, SAVE_TOP)
print("\nTF-IDF baseline on dev:", {k: round(v, 4) for k, v in retrieval_metrics(ranked_tfidf, dev_split.gold_rows).items()})
print("first dev query (start):", dev_split.queries.texts[0][:300])


== train: 817 queries | pool 4320 documents
  tokens/query percentiles (50/75/90/95/99/max): [  5864   9994  16178  26251  57411 152238]
  tokens/pool-doc percentiles:                  [  4644   8603  15712  24667  62120 526615]
  pool docs that fit entirely in 4 chunks: 20.1% | median chunks/doc 10
  queries whose own copy is in their pool: 817
  relevant/query percentiles (min/25/50/75/90/max): [ 1  2  4  9 14 52]
  share of the pool that is relevant to a query: 0.1504% | queries with more than 7 relevant: 30.4%
  upper bound of Micro-F1@7: 0.6426
  most frequently relevant pool docs (count): [14, 10, 10, 9, 8] | distinct relevant docs: 4107

== dev: 118 queries | pool 1023 documents
  tokens/query percentiles (50/75/90/95/99/max): [  5701  10153  19773  25130 129598 133946]
  tokens/pool-doc percentiles:                  [  5145   9612  21546  34535 108109 526615]
  pool docs that fit entirely in 4 chunks: 18.4% | median chunks/doc 11
  queries whose own copy is in their pool: 118


## 7. Finetuning methods

In [10]:
class LoRALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A x)   (additive low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A(Wx))  ==  (I + s*BA) W x   (multiplicative low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

class QLoRALinear(nn.Module):
    """Frozen weight stored as int8 (per-output-channel absmax quantisation, dequantised on the fly) + trainable LoRA.
    Weight-only 8-bit simulation of QLoRA (no bitsandbytes needed)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        assert base.bias is None, "QLoRALinear expects bias-free projections"
        weight = base.weight.detach()
        scale = weight.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
        self.register_buffer("w_int8", torch.round(weight / scale).clamp(-127, 127).to(torch.int8))
        self.register_buffer("w_scale", scale)
        self.lora_scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        dequantised = self.w_int8.float() * self.w_scale
        return F.linear(x, dequantised) + self.lora_scale * F.linear(F.linear(x, self.A), self.B)

class IA3Linear(nn.Module):
    """IA3: the output of a frozen linear layer is rescaled by a learned per-feature vector (initialised to 1 = identity)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        return self.base(x) * self.scale

class SoftPrefix(nn.Module):
    """n learned vectors prepended to the embeddings of every chunk (backbone stays frozen).
    reparam=False -> prompt tuning; reparam=True -> prefix-style MLP reparameterisation of the vectors."""
    def __init__(self, n_tokens, dim, token_embeddings, reparam):
        super().__init__()
        init_rows = torch.randint(0, token_embeddings.size(0), (n_tokens,))
        self.latent = nn.Parameter(token_embeddings[init_rows].clone())
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Linear(dim, dim)) if reparam else None
    def forward(self):
        return self.latent if self.mlp is None else self.mlp(self.latent)

PEFT_WRAPPERS = {"lora": LoRALinear, "lorma": LoRMALinear, "qlora": QLoRALinear}
ATTN_PROJ_NAMES = ("q_proj", "k_proj", "v_proj", "out_proj", "kv_down_proj", "k_up_proj", "v_up_proj")  # GQA and MLA names

def wrap_attention_projections(backbone, wrapper_cls, rank, alpha):
    """Wrap every attention projection that exists in this architecture (GQA / MLA / conv variants)."""
    for block in backbone.blocks:
        for name in ATTN_PROJ_NAMES:
            if hasattr(block.attn, name):
                setattr(block.attn, name, wrapper_cls(getattr(block.attn, name), rank, alpha))

def apply_ia3(backbone):
    """IA3 on the key / value projections and on the FFN intermediate activations (the `up` branch of every SwiGLU expert, or the first FFN layer)."""
    for block in backbone.blocks:
        for name in ("k_proj", "v_proj", "k_up_proj", "v_up_proj"):
            if hasattr(block.attn, name):
                setattr(block.attn, name, IA3Linear(getattr(block.attn, name)))
        if hasattr(block.ffn, "experts"):
            for expert in block.ffn.experts:
                expert.up = IA3Linear(expert.up)
        else:
            block.ffn.net[0] = IA3Linear(block.ffn.net[0])

## 8. Data and Model Utilities

In [11]:
def encode_chunks(backbone, ids, use_ckpt, prefix_emb):
    """Phase 1 decoder over [N, L] chunks (causal, right-padded) -> hidden states [N, L, D].
    Right padding + causal attention means padding never influences real tokens, so no mask is needed.
    Optional soft-prefix vectors [n, D] are prepended to every chunk and removed from the output."""
    N, L = ids.shape
    x = backbone.tok_emb(ids) * backbone.emb_scale
    n_prefix = 0
    if prefix_emb is not None:
        n_prefix = prefix_emb.size(0)
        x = torch.cat([prefix_emb.unsqueeze(0).expand(N, -1, -1).to(x.dtype), x], dim=1)
    total_len = n_prefix + L
    position_ids = torch.arange(total_len, device=ids.device).unsqueeze(0).expand(N, total_len)
    if backbone.use_sin:
        x = x + backbone.pos_emb[position_ids]
    for block in backbone.blocks:
        if use_ckpt and torch.is_grad_enabled():
            x = checkpoint(lambda h, blk=block: blk(h, position_ids, None)[0], x, use_reentrant=False)
        else:
            x = block(x, position_ids, None)[0]
    return backbone.ln_f(x)[:, n_prefix:]

# ── pooling of chunk vectors [B, C, D] into one document vector [B, D] ──
class MeanPooler(nn.Module):
    def forward(self, grid, mask):
        m = mask.unsqueeze(-1).to(grid.dtype)
        return (grid * m).sum(1) / m.sum(1)

class AttentionPooler(nn.Module):
    """Learned attention over chunks."""
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)
    def forward(self, grid, mask):
        scores = self.score(grid).squeeze(-1).float().masked_fill(~mask, float("-inf"))
        return (torch.softmax(scores, dim=1).unsqueeze(-1) * grid.float()).sum(1)

POOLERS = {"mean": lambda d: MeanPooler(), "attn": AttentionPooler}

class RetrievalEncoder(nn.Module):
    """Phase 1 decoder -> one embedding per chunk -> (mean / attention) document vector, or the chunk grid for maxsim."""
    def __init__(self, backbone, soft_prefix, chunk_pool, token_pool, use_ckpt):
        super().__init__()
        self.backbone, self.soft_prefix, self.token_pool, self.use_ckpt = backbone, soft_prefix, token_pool, use_ckpt
        self.chunk_pooler = POOLERS[chunk_pool](backbone.cfg.d_model)

    def chunk_embeddings(self, ids, chunk_mask):
        B, C, L = ids.shape
        flat_ids = ids[chunk_mask]                                   # only real chunks: [N, L]
        prefix = self.soft_prefix() if self.soft_prefix is not None else None
        hidden = encode_chunks(self.backbone, flat_ids, self.use_ckpt, prefix)
        real = flat_ids.ne(PAD)
        if self.token_pool == "mean":                                # masked mean over the chunk's tokens
            weight = real.unsqueeze(-1).to(hidden.dtype)
            embedding = (hidden * weight).sum(1) / weight.sum(1).clamp(min=1.0)
        else:                                                        # hidden state of the final </s> token
            assert self.token_pool == "last", self.token_pool
            embedding = hidden[torch.arange(hidden.size(0), device=hidden.device), real.sum(1) - 1]
        grid = embedding.new_zeros(B, C, embedding.size(-1))
        grid[chunk_mask] = embedding
        return grid.float()                                          # [B, C, D]

    def forward(self, ids, chunk_mask, similarity):
        grid = self.chunk_embeddings(ids, chunk_mask)
        if similarity == "maxsim":                                   # late interaction: keep every chunk vector
            return {"grid": F.normalize(grid, dim=-1), "mask": chunk_mask}
        return {"vec": F.normalize(self.chunk_pooler(grid, chunk_mask).float(), dim=-1)}

# ── model factory ──
def freeze(module):
    for p in module.parameters():
        p.requires_grad = False

def build_model(base_lm, exp):
    assert exp.similarity != "maxsim" or exp.chunk_pool == "mean", "maxsim uses the chunk grid directly; chunk_pool must stay 'mean'"
    assert exp.teacher_name is None or exp.similarity == "cosine", "distillation needs pooled (cosine) embeddings"
    assert exp.method != "frozen" or exp.chunk_pool == "mean", "a frozen model has no trained pooler"
    backbone = copy.deepcopy(base_lm)
    if exp.block_stride > 1:                                         # KD student: keep every n-th block
        keep = list(range(0, len(backbone.blocks), exp.block_stride))
        backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in keep])
    soft_prefix = None

    if exp.method == "full":
        pass                                                         # everything trainable
    elif exp.method == "frozen":                                     # zero-shot Phase 1 representations
        freeze(backbone)
    elif exp.method == "partial":                                    # top-n blocks + final norm
        freeze(backbone)
        for block in backbone.blocks[-exp.n_top:]:
            for p in block.parameters(): p.requires_grad = True
        for p in backbone.ln_f.parameters(): p.requires_grad = True
    elif exp.method in PEFT_WRAPPERS:
        freeze(backbone)
        wrap_attention_projections(backbone, PEFT_WRAPPERS[exp.method], exp.rank, exp.alpha)
    elif exp.method == "ia3":
        freeze(backbone)
        apply_ia3(backbone)
    elif exp.method in ("prompt", "prefix"):
        freeze(backbone)
        soft_prefix = SoftPrefix(exp.n_prefix, backbone.cfg.d_model, backbone.tok_emb.weight.detach(), reparam=(exp.method == "prefix"))
    else:
        raise ValueError(f"unknown method: {exp.method}")
    return RetrievalEncoder(backbone, soft_prefix, exp.chunk_pool, exp.token_pool, GRAD_CKPT).to(DEVICE)

def count_params(model):
    """(total, trainable) in millions. Quantised int8 weights (QLoRA) are counted as parameters."""
    total = sum(p.numel() for p in model.parameters()) + sum(b.numel() for n, b in model.named_buffers() if n.endswith("w_int8"))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total / 1e6, trainable / 1e6

def model_size_mb(model):
    """Real storage of the weights (int8 weights of QLoRA count 1 byte)."""
    n_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    n_bytes += sum(b.numel() * b.element_size() for n, b in model.named_buffers() if n.endswith(("w_int8", "w_scale")))
    return n_bytes / 2**20

In [12]:
def similarity_matrix(query_rep, doc_rep, similarity):
    """[n_queries, n_docs] similarity. cosine: dot of normalised document vectors. maxsim: every query chunk is matched
    to its best document chunk and the matches are averaged (late interaction)."""
    if similarity == "cosine":
        return query_rep["vec"] @ doc_rep["vec"].T
    assert similarity == "maxsim", similarity
    sims = torch.einsum("qcd,ned->qnce", query_rep["grid"], doc_rep["grid"])           # [Q, N, Cq, Cd]
    sims = sims.masked_fill(~doc_rep["mask"][None, :, None, :], -2.0)
    best = sims.max(dim=-1).values                                                     # [Q, N, Cq]
    weight = query_rep["mask"][:, None, :].to(best.dtype)
    return (best * weight).sum(-1) / weight.sum(-1)

def ranking_loss(scores, target, not_negative, exp):
    """scores [Q, D]: every query against every document of the batch; target[q] = column of its positive."""
    rows = torch.arange(scores.size(0), device=scores.device)
    if exp.loss_type == "infonce":                                   # softmax over positive + negatives (listwise)
        usable = not_negative.clone()
        usable[rows, target] = False                                 # the positive itself stays in the softmax
        return F.cross_entropy((scores / exp.temperature).masked_fill(usable, float("-inf")), target)
    assert exp.loss_type == "triplet", exp.loss_type                 # margin loss against the hardest negative (pairwise)
    positive = scores[rows, target]
    hardest_negative = scores.masked_fill(not_negative, float("-inf")).max(dim=1).values
    return F.relu(exp.margin - positive + hardest_negative).mean()

def distillation_losses(student_q, student_d, scores, batch, teacher, exp):
    """KD from precomputed teacher embeddings: (1) match the teacher's query->document score distribution (KL),
    (2) pull the student's embeddings towards the teacher's (cosine)."""
    teacher_q, teacher_d = teacher["q"][batch["q_rows"]], teacher["d"][batch["d_rows"]]
    teacher_scores = teacher_q @ teacher_d.T
    kd_scores = F.kl_div(F.log_softmax(scores / exp.kd_temperature, -1), F.softmax(teacher_scores / exp.kd_temperature, -1), reduction="batchmean")
    kd_embed = (1 - (student_q["vec"] * teacher_q).sum(-1)).mean() + (1 - (student_d["vec"] * teacher_d).sum(-1)).mean()
    return kd_scores, kd_embed

@torch.no_grad()
def encode_corpus(model, docset, exp):
    """Embed every document of a DocSet once -> dict of CPU tensors (cosine: 'vec' [N, D]; maxsim: 'grid' [N, C, D] + 'mask')."""
    model.eval()
    loader = DataLoader(DocDataset(docset, exp.strategy, exp.max_chunks), batch_size=EVAL_BATCH_DOCS, shuffle=False,
                        collate_fn=lambda items: pad_chunks(items, exp.max_chunks))
    parts = []
    for ids, chunk_mask in tqdm(loader, desc="encode", leave=False):
        with autocast("cuda", dtype=torch.float16):
            rep = model(ids.to(DEVICE), chunk_mask.to(DEVICE), exp.similarity)
        parts.append({k: v.cpu() for k, v in rep.items()})
    return {k: torch.cat([p[k] for p in parts]) for k in parts[0]}

@torch.no_grad()
def score_queries(query_rep, pool_rep, similarity):
    pool_gpu = {k: v.to(DEVICE) for k, v in pool_rep.items()}
    n_queries = next(iter(query_rep.values())).size(0)
    blocks = []
    for start in range(0, n_queries, QUERY_BLOCK):
        block = {k: v[start: start + QUERY_BLOCK].to(DEVICE) for k, v in query_rep.items()}
        blocks.append(similarity_matrix(block, pool_gpu, similarity).float().cpu())
    return torch.cat(blocks).numpy()

def evaluate_split(model, exp, split):
    """Encode queries + pool, score every query against its own pool, rank. Returns scores [n_queries, pool], top-SAVE_TOP rows, timings."""
    torch.cuda.synchronize(); t0 = time.perf_counter()
    query_rep, pool_rep = encode_corpus(model, split.queries, exp), encode_corpus(model, split.pool, exp)
    torch.cuda.synchronize(); encode_sec = time.perf_counter() - t0
    t0 = time.perf_counter()
    scores = score_queries(query_rep, pool_rep, exp.similarity)
    ranked = rank_from_scores(scores, split.self_rows, SAVE_TOP)
    out = {"scores": scores, "ranked": ranked, "encode_ms_per_doc": 1000 * encode_sec / (len(split.queries.ids) + len(split.pool.ids)),
           "score_ms_per_query": 1000 * (time.perf_counter() - t0) / len(split.queries.ids),
           "index_MB": sum(v.numel() * (2 if v.is_floating_point() else 1) for v in pool_rep.values()) / 2**20}   # pool index stored in fp16
    if split.gold_rows is not None:
        out["metrics"] = retrieval_metrics(ranked, split.gold_rows)
    return out

## 9. Training Utilities

In [13]:
def checkpoint_path(name):
    return f"{OUT}/ckpt_{name}.pt"     # latest epoch (resume point), overwritten every epoch

def best_path(name):
    return f"{OUT}/best_{name}.pt"     # best epoch only, kept separately

def save_atomic(obj, path):
    """Write to a temp file, then rename: an interruption can never corrupt the previous file."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}

def set_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])

def save_best(exp, best_state, best_metric, epoch):
    save_atomic({"state": best_state, "metric": best_metric, "epoch": epoch}, best_path(exp.name))

def save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, epoch, global_step, history):
    """Everything needed to continue training from the end of `epoch` (epochs completed so far)."""
    model_state = {k: v.detach().cpu() for k, v in model.state_dict().items() if k in snapshot_keys}
    save_atomic({"config": vars(exp), "epoch": epoch, "global_step": global_step, "history": history, "model": model_state,
                 "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(), "rng": get_rng_state()}, checkpoint_path(exp.name))

MICRO_BATCH_FIELDS = {"batch_queries", "grad_accum"}   # may change between runs as long as their product is unchanged

def resume_if_checkpointed(exp, model, optimizer, scaler, fresh_history, worst_metric):
    """Returns (start_epoch, global_step, history, best_metric, best_state); fresh values when there is no checkpoint yet."""
    if not os.path.exists(checkpoint_path(exp.name)):
        return 0, 0, fresh_history, worst_metric, None

    ckpt = torch.load(checkpoint_path(exp.name), map_location="cpu", weights_only=False)
    saved = {k: v for k, v in ckpt["config"].items() if k not in MICRO_BATCH_FIELDS}
    current = {k: v for k, v in vars(exp).items() if k not in MICRO_BATCH_FIELDS}
    assert saved == current, f"{checkpoint_path(exp.name)} was written by a different configuration; delete it (and best_{exp.name}.pt) to start over"
    assert ckpt["config"]["batch_queries"] * ckpt["config"]["grad_accum"] == exp.batch_queries * exp.grad_accum, "effective batch size changed"
    outcome = model.load_state_dict(ckpt["model"], strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    set_rng_state(ckpt["rng"])

    best = torch.load(best_path(exp.name), map_location="cpu", weights_only=False)
    print(f"[{exp.name}] resumed after epoch {ckpt['epoch']}/{exp.epochs} (best so far {best['metric']:.4f} at epoch {best['epoch']})")
    return ckpt["epoch"], ckpt["global_step"], ckpt["history"], best["metric"], best["state"]

In [14]:
def snapshot_names(model):
    """Parameters that must be restored to recover a given epoch (frozen weights never change)."""
    return {n for n, p in model.named_parameters() if p.requires_grad}

def train_model(model, exp, train_split, dev_split, teacher):
    dataset = PairDataset(train_split, exp)
    loader = DataLoader(dataset, batch_size=exp.batch_queries, shuffle=True, collate_fn=PairBatcher(train_split, exp), pin_memory=True)

    # backbone adaptation (LoRA / unfrozen layers / soft prefix) and the new pooler get different learning rates
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    assert trainable, "no trainable parameters"
    is_body = lambda name: name.startswith(("backbone.", "soft_prefix."))
    groups = [(exp.lr, [p for n, p in trainable if is_body(n)], WEIGHT_DECAY),
              (exp.head_lr, [p for n, p in trainable if not is_body(n)], 0.0)]
    groups = [g for g in groups if g[1]]
    optimizer = torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd} for lr, params, wd in groups])
    base_lrs = [lr for lr, _, _ in groups]

    steps_per_epoch = math.ceil(len(loader) / exp.grad_accum)
    total_steps = exp.epochs * steps_per_epoch
    warmup_steps = int(WARMUP_FRAC * total_steps)
    scaler = GradScaler("cuda")

    fresh_history = {"val_f1": [], "val_P": [], "val_R": [], "val_R50": [], "loss": [], "train_sec": 0.0}
    start_epoch, global_step, history, best_f1, best_state = resume_if_checkpointed(exp, model, optimizer, scaler, fresh_history, worst_metric=-1.0)

    for epoch in range(start_epoch, exp.epochs):
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0

        for step_in_epoch, batch in enumerate(tqdm(loader, desc=f"{exp.name} ep {epoch + 1}/{exp.epochs}", leave=False)):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast("cuda", dtype=torch.float16):
                query_rep = model(batch["q_ids"], batch["q_mask"], exp.similarity)
                doc_rep = model(batch["d_ids"], batch["d_mask"], exp.similarity)
            scores = similarity_matrix(query_rep, doc_rep, exp.similarity)          # [queries, docs in batch], float32

            loss = ranking_loss(scores, batch["target"], batch["not_negative"], exp)
            if teacher is not None:                                                # knowledge distillation
                kd_scores, kd_embed = distillation_losses(query_rep, doc_rep, scores, batch, teacher, exp)
                loss = loss + exp.kd_score_weight * kd_scores + exp.kd_embed_weight * kd_embed

            scaler.scale(loss / exp.grad_accum).backward()
            epoch_loss += loss.item()

            is_boundary = (step_in_epoch + 1) % exp.grad_accum == 0 or step_in_epoch == len(loader) - 1
            if is_boundary:
                # linear warm-up, then cosine decay
                if global_step < warmup_steps:
                    lr_frac = (global_step + 1) / max(1, warmup_steps)
                else:
                    progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
                    lr_frac = 0.5 * (1.0 + math.cos(math.pi * progress))
                for group, base_lr in zip(optimizer.param_groups, base_lrs):
                    group["lr"] = base_lr * lr_frac

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

        # ── validation: model selection by dev Micro-F1@7 ──
        metrics = evaluate_split(model, exp, dev_split)["metrics"]
        history["val_f1"].append(metrics["F1@7"]); history["val_P"].append(metrics["P@7"])
        history["val_R"].append(metrics["R@7"]);   history["val_R50"].append(metrics["R@50"])
        history["loss"].append(epoch_loss / len(loader)); history["train_sec"] += time.time() - t0
        print(f"[{exp.name}] ep {epoch + 1}/{exp.epochs} loss {history['loss'][-1]:.4f} | dev F1@7 {metrics['F1@7']:.4f} "
              f"P@7 {metrics['P@7']:.3f} R@7 {metrics['R@7']:.3f} R@50 {metrics['R@50']:.3f}")

        # ── checkpoints: best model (only when improved) first, then the resume point ──
        snapshot_keys = snapshot_names(model)
        if metrics["F1@7"] > best_f1:
            best_f1 = metrics["F1@7"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in snapshot_keys}
            save_best(exp, best_state, best_f1, epoch + 1)
        save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, epoch + 1, global_step, history)

    model.load_state_dict(best_state, strict=False)                  # restore best epoch
    history["best_f1"] = best_f1
    return history

In [15]:
@dataclass
class Experiment:
    name: str
    base_model: str               # one of PHASE1_CONFIGS
    method: str                   # full | partial | lora | lorma | qlora | ia3 | prompt | prefix | frozen
    similarity: str               # cosine | maxsim
    chunk_pool: str               # mean | attn   (chunks -> document vector, cosine only)
    token_pool: str               # mean | last   (tokens -> chunk vector)
    strategy: str                 # head | tail | head_tail | uniform
    max_chunks: int               # chunks per document
    loss_type: str                # infonce | triplet
    hard_neg: str                 # none | random | tfidf
    n_hard: int                   # hard negatives per query
    temperature: float            # InfoNCE temperature
    margin: float                 # triplet margin
    epochs: int
    lr: float                     # backbone / adapter / prefix learning rate
    head_lr: float                # chunk-pooler learning rate
    batch_queries: int
    grad_accum: int
    rank: int                     # LoRA / LoRMA / QLoRA rank
    alpha: float
    n_top: int                    # blocks trained by "partial"
    n_prefix: int                 # soft tokens for prompt / prefix tuning
    block_stride: int             # 1 = all blocks; 2 = every 2nd block (KD student)
    teacher_name: str             # experiment whose embeddings are distilled (None = no KD)
    kd_temperature: float
    kd_score_weight: float        # weight of the KL term on the teacher's similarity distribution
    kd_embed_weight: float        # weight of the embedding-matching term

BASE = Experiment(name="", base_model=MAIN_MODEL, method="lorma", similarity="cosine", chunk_pool="mean", token_pool="mean",
                  strategy="uniform", max_chunks=DEFAULT_MAX_CHUNKS, loss_type="infonce", hard_neg="tfidf", n_hard=1,
                  temperature=0.05, margin=0.2, epochs=3, lr=2e-4, head_lr=1e-3, batch_queries=BATCH_QUERIES, grad_accum=GRAD_ACCUM,
                  rank=16, alpha=32.0, n_top=4, n_prefix=16, block_stride=1, teacher_name=None,
                  kd_temperature=0.1, kd_score_weight=1.0, kd_embed_weight=1.0)

def make_exp(name, **overrides):
    return replace(BASE, name=name, **overrides)

def result_path(name):
    return f"{OUT}/result_{name}.json"

def load_trained_model(name):
    """Rebuild a finished experiment from its saved config + best-epoch weights."""
    exp = Experiment(**json.load(open(result_path(name)))["config"])
    model = build_model(load_phase1_model(exp.base_model), exp)
    best = torch.load(best_path(name), map_location="cpu", weights_only=False)
    outcome = model.load_state_dict(best["state"], strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    return model, exp

# ── knowledge distillation: teacher embeddings of every train query / pool document ──
def teacher_embeddings_path(name):
    return f"{OUT}/teacher_emb_{name}.pt"

def ensure_teacher_embeddings(teacher_name):
    """Exports the teacher's train-set embeddings from its best weights if that was not done yet (no retraining)."""
    if os.path.exists(teacher_embeddings_path(teacher_name)):
        return
    model, exp = load_trained_model(teacher_name)
    assert exp.similarity == "cosine", "the teacher must be a cosine (pooled) model"
    torch.save({"query_ids": train_split.queries.ids, "pool_ids": train_split.pool.ids,
                "q": encode_corpus(model, train_split.queries, exp)["vec"].half(),
                "d": encode_corpus(model, train_split.pool, exp)["vec"].half()}, teacher_embeddings_path(teacher_name))
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

def load_teacher_embeddings(teacher_name):
    saved = torch.load(teacher_embeddings_path(teacher_name), weights_only=False)
    assert saved["query_ids"] == train_split.queries.ids and saved["pool_ids"] == train_split.pool.ids, "teacher embeddings belong to another train set"
    return {"q": saved["q"].to(DEVICE).float(), "d": saved["d"].to(DEVICE).float()}

# ── submission ──
def write_submission(ranked_rows, name):
    """prediction.json = {query id: [7 candidate ids, best first]}, candidates only from the query's own test pool."""
    pool_ids = test_split.pool.ids
    submission = {qid: [pool_ids[int(r)] for r in row[:TOP_K]] for qid, row in zip(test_split.queries.ids, ranked_rows)}
    assert len(submission) == len(test_split.queries.ids)
    for qid, picks in submission.items():
        assert len(picks) == TOP_K and len(set(picks)) == TOP_K and set(picks) <= set(pool_ids), f"invalid prediction for {qid}"
    json_path, zip_path = f"{OUT}/submission_{name}.json", f"{OUT}/submission_{name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, "prediction.json")
    print(f"  -> {zip_path}")

def run_experiment(exp):
    """Build -> train -> evaluate dev (timed) -> rank test -> write submission -> save results. Skips finished experiments."""
    if os.path.exists(result_path(exp.name)):
        print(f"[{exp.name}] finished earlier -> skipped"); return

    set_seed(SEED)
    model = build_model(load_phase1_model(exp.base_model), exp)
    teacher = None
    if exp.teacher_name is not None:
        ensure_teacher_embeddings(exp.teacher_name)
        teacher = load_teacher_embeddings(exp.teacher_name)

    total_m, trainable_m = count_params(model)
    docs_per_step = exp.batch_queries * (2 + exp.n_hard)
    print(f"\n=== {exp.name}: total {total_m:.1f}M | trainable {trainable_m:.2f}M ({100 * trainable_m / total_m:.1f}%) "
          f"| up to {docs_per_step * exp.max_chunks} chunks of {CHUNK_LEN} tokens per micro-batch")
    torch.cuda.reset_peak_memory_stats()
    history = train_model(model, exp, train_split, dev_split, teacher)

    dev, test = evaluate_split(model, exp, dev_split), evaluate_split(model, exp, test_split)
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    print(f"  best-epoch dev F1@7 during training {history['best_f1']:.4f} | re-evaluated {dev['metrics']['F1@7']:.4f}")
    np.save(f"{OUT}/dev_scores_{exp.name}.npy", dev["scores"]); np.save(f"{OUT}/test_scores_{exp.name}.npy", test["scores"])
    write_submission(test["ranked"], exp.name)

    result = {"name": exp.name, "config": asdict(exp), "total_M": total_m, "trainable_M": trainable_m, "size_MB": model_size_mb(model),
              "dev": dev["metrics"], "encode_ms_per_doc": dev["encode_ms_per_doc"], "score_ms_per_query": dev["score_ms_per_query"],
              "index_MB": dev["index_MB"], "train_min": history["train_sec"] / 60, "peak_GB": peak_gb, "history": history}
    with open(result_path(exp.name), "w") as f:
        json.dump(result, f, indent=2)
    m = dev["metrics"]
    print(f"### {exp.name}: trainable {trainable_m:.2f}M/{total_m:.1f}M | dev F1@7 {m['F1@7']:.4f} P@7 {m['P@7']:.3f} R@7 {m['R@7']:.3f} "
          f"| R@20 {m['R@20']:.3f} R@50 {m['R@50']:.3f}")

    model.cpu(); del model, teacher; gc.collect(); torch.cuda.empty_cache()

## 10. Sanity Check

In [16]:
lm = load_phase1_model(MAIN_MODEL).to(DEVICE).eval()
x = torch.tensor([[BOS] + train_split.pool.token_ids[0][:CONTENT_LEN].tolist()], device=DEVICE)
with torch.no_grad():
    logits = lm(x[:, :-1])
lm_loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1)).item()
print(f"Phase-1 LM loss on a train document: {lm_loss:.3f} (random init ~ {math.log(VOCAB_SIZE):.1f})")
del lm; gc.collect(); torch.cuda.empty_cache()

Phase-1 LM loss on a train document: 2.833 (random init ~ 9.9)


In [17]:
def zero_shot_baseline(model_name, **overrides):
    """Frozen Phase 1 representations, no training: how much retrieval ability does pretraining alone provide?"""
    exp = make_exp(f"zero_shot_{model_name}", base_model=model_name, method="frozen", **overrides)
    model = build_model(load_phase1_model(model_name), exp)
    metrics = evaluate_split(model, exp, dev_split)["metrics"]
    print(f"zero-shot Phase 1 ({model_name}, {exp.token_pool}-pooled): " + ", ".join(f"{k} {v:.4f}" for k, v in metrics.items()))
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

for model_name in PHASE1_CONFIGS:
    zero_shot_baseline(model_name)

zero-shot Phase 1 (gqa-rope, mean-pooled): F1@7 0.2212, P@7 0.2409, R@7 0.2045, R@20 0.3731, R@50 0.5149


zero-shot Phase 1 (mla-sin, mean-pooled): F1@7 0.1501, P@7 0.1634, R@7 0.1387, R@20 0.2909, R@50 0.4378


zero-shot Phase 1 (gqa-rope-swiglu-moe, mean-pooled): F1@7 0.2668, P@7 0.2906, R@7 0.2467, R@20 0.4039, R@50 0.5427


zero-shot Phase 1 (swiglu-moe-rope-conv, mean-pooled): F1@7 0.2101, P@7 0.2288, R@7 0.1942, R@20 0.3690, R@50 0.5149


In [18]:
if SMOKE_TEST:
    quick = dict(epochs=1, batch_queries=2, grad_accum=2, max_chunks=2)
    run_experiment(make_exp("smoke_full", method="full", **quick))
    run_experiment(make_exp("smoke_kd", method="full", block_stride=2, teacher_name="smoke_full", **quick))
    for method in ["partial", "lora", "lorma", "qlora", "ia3", "prompt", "prefix"]:
        run_experiment(make_exp(f"smoke_{method}", method=method, **quick))
    for model_name in PHASE1_CONFIGS:
        run_experiment(make_exp(f"smoke_lorma_{model_name}", base_model=model_name, **quick))
    run_experiment(make_exp("smoke_maxsim", similarity="maxsim", **quick))
    run_experiment(make_exp("smoke_attn_last", chunk_pool="attn", token_pool="last", **quick))
    run_experiment(make_exp("smoke_triplet", loss_type="triplet", **quick))
    for hard_neg in ["none", "random"]:
        run_experiment(make_exp(f"smoke_neg_{hard_neg}", hard_neg=hard_neg, **quick))
    for strategy in ["head", "tail", "head_tail"]:
        run_experiment(make_exp(f"smoke_strategy_{strategy}", strategy=strategy, **quick))

## 12. Run Experiments

In [19]:
run_experiment(make_exp("full_finetune", method="full", epochs=3, lr=3e-5, head_lr=5e-4, batch_queries=2, grad_accum=8))


=== full_finetune: total 190.0M | trainable 190.02M (100.0%) | up to 24 chunks of 512 tokens per micro-batch


[full_finetune] ep 1/3 loss 1.0144 | dev F1@7 0.2768 P@7 0.301 R@7 0.256 R@50 0.563


[full_finetune] ep 2/3 loss 0.8010 | dev F1@7 0.2857 P@7 0.311 R@7 0.264 R@50 0.583


[full_finetune] ep 3/3 loss 0.7154 | dev F1@7 0.2868 P@7 0.312 R@7 0.265 R@50 0.573


  best-epoch dev F1@7 during training 0.2868 | re-evaluated 0.2868
  -> ./cs781_phase7/submission_full_finetune.zip
### full_finetune: trainable 190.02M/190.0M | dev F1@7 0.2868 P@7 0.312 R@7 0.265 | R@20 0.444 R@50 0.573


In [24]:
# for model_name in PHASE1_CONFIGS:
#     run_experiment(make_exp(f"lorma_{model_name}", base_model=model_name, method="lorma", lr=2e-4))
run_experiment(make_exp("lora_r16",  base_model= "mla-sin",   method="lora", epochs=1,     lr=2e-4))
run_experiment(make_exp("qlora_r16",    method="qlora",    lr=2e-4))
run_experiment(make_exp("ia3", base_model=  "swiglu-moe-rope-conv", epochs=1,  method="ia3",  lr=5e-4, head_lr=5e-3))
run_experiment(make_exp("prompt_16",  base_model= "gqa-rope",   method="prompt", epochs=1,   lr=3e-3, n_prefix=16))
run_experiment(make_exp("kd_student_half_depth", method="full", block_stride=2, teacher_name="full_finetune",
                        epochs=1, lr=5e-5, head_lr=5e-4, batch_queries=2, grad_accum=8,
                        kd_temperature=0.1, kd_score_weight=1.0, kd_embed_weight=1.0))

[lora_r16] finished earlier -> skipped
[qlora_r16] finished earlier -> skipped
[ia3] finished earlier -> skipped
[prompt_16] finished earlier -> skipped



=== kd_student_half_depth: total 102.7M | trainable 102.69M (100.0%) | up to 24 chunks of 512 tokens per micro-batch


[kd_student_half_depth] ep 1/1 loss 1.4401 | dev F1@7 0.2368 P@7 0.258 R@7 0.219 R@50 0.546


  best-epoch dev F1@7 during training 0.2368 | re-evaluated 0.2368
  -> ./cs781_phase7/submission_kd_student_half_depth.zip
### kd_student_half_depth: trainable 102.69M/102.7M | dev F1@7 0.2368 P@7 0.258 R@7 0.219 | R@20 0.382 R@50 0.546
